# Deploy pipeline: load, preprocess, train, and export model

This notebook loads `data.csv`, drops the columns not used by the app, trains a `StandardScaler` + `RandomForestClassifier` pipeline, and writes `multiclass_classification_model.pkl` and `final_feature_list.txt` to the project root so `app.py` can load them directly.

**Note on `data.csv`:** the file mixes delimiters (the first rows use commas, the rest use semicolons). The loader below normalises every line to commas so all 4,424 students are used.

In [1]:
# 1) Imports
import io
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

print('Imports OK')

Imports OK


In [2]:
# 2) Load data.csv, normalising mixed comma/semicolon delimiters
csv_path = 'data.csv'
assert os.path.exists(csv_path), f'{csv_path} not found in repo root'

with open(csv_path, 'r', encoding='utf-8-sig') as f:
    text = f.read()

# Some rows use ';' and others ',' - convert everything to ','
text = text.replace(';', ',')

df = pd.read_csv(io.StringIO(text))
print('Data loaded. Shape:', df.shape)

Data loaded. Shape: (4424, 37)


In [3]:
# 3) Clean column names and encode the target
df.columns = df.columns.str.strip()
if 'Target' not in df.columns:
    raise RuntimeError('Target column not found in data.csv')

# Same encoding as app.py: Enrolled = 0, Dropout = 1, Graduate = 2
target_map = {'Enrolled': 0, 'Dropout': 1, 'Graduate': 2}
df['Target'] = df['Target'].str.strip()
df['Target_encoded'] = df['Target'].map(target_map)
print(df['Target'].value_counts())

Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


In [4]:
# 4) Drop the columns the app does not use
features_to_drop = [
    'Application mode', 'Application order', 'Previous qualification', 'Previous qualification (grade)',
    "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation",
    'Admission grade', 'Debtor', 'International', 'Curricular units 1st sem (credited)',
    'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (without evaluations)',
    'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (evaluations)',
    'Curricular units 2nd sem (without evaluations)',
]

X = df.drop(columns=['Target', 'Target_encoded'] + features_to_drop)
y = df['Target_encoded']
print('Remaining features:', X.shape[1])
print(X.columns.tolist())

Remaining features: 19
['Marital status', 'Course', 'Daytime/evening attendance', 'Nacionality', 'Displaced', 'Educational special needs', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Unemployment rate', 'Inflation rate', 'GDP']


In [5]:
# 5) Check types and missing values
X = X.apply(pd.to_numeric, errors='coerce')
missing = X.isnull().sum().sum()
print('Missing values:', missing)
if missing > 0:
    keep = X.notnull().all(axis=1)
    X, y = X[keep], y[keep]
    print('Rows after dropping missing values:', len(X))

Missing values: 0


In [6]:
# 6) Evaluate on a stratified 20% hold-out set, then retrain on all data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train/Test shapes:', X_train.shape, X_test.shape)

def make_pipeline():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, class_weight='balanced')),
    ])

pipeline = make_pipeline()
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print('Validation accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Macro F1:', round(f1_score(y_test, y_pred, average='macro'), 4))
print(classification_report(y_test, y_pred, target_names=['Enrolled', 'Dropout', 'Graduate']))

# Refit on the full dataset for the deployed model
pipeline_full = make_pipeline()
pipeline_full.fit(X, y)
print('Trained pipeline on full data.')

Train/Test shapes: (3539, 19) (885, 19)


Validation accuracy: 0.739
Macro F1: 0.696
              precision    recall  f1-score   support

    Enrolled       0.43      0.60      0.50       159
     Dropout       0.86      0.67      0.75       284
    Graduate       0.83      0.83      0.83       442

    accuracy                           0.74       885
   macro avg       0.71      0.70      0.70       885
weighted avg       0.77      0.74      0.75       885



Trained pipeline on full data.


In [7]:
# 7) Save the pipeline and feature list for app.py
model_path = 'multiclass_classification_model.pkl'
joblib.dump(pipeline_full, model_path)
print('Saved model to', model_path)

with open('final_feature_list.txt', 'w', encoding='utf-8') as f:
    for feat in X.columns:
        f.write(f'{feat}\n')
print('Saved final_feature_list.txt with', len(X.columns), 'features')

Saved model to multiclass_classification_model.pkl
Saved final_feature_list.txt with 19 features


In [8]:
# 8) Quick load test
loaded = joblib.load(model_path)
sample = X.iloc[[0]]
print('Classes:', loaded.classes_)
print('Prediction for first student:', loaded.predict(sample)[0])
print('Probabilities:', loaded.predict_proba(sample)[0].round(3))

Classes: [0 1 2]
Prediction for first student: 1
Probabilities: [0.274 0.516 0.21 ]
